# Layer 18 PLS scores

Downloads `layer_out/18` at prompt position **-1**, fits a **6-component PLS regression** against
`log10_time_horizon_months`, fits a **3-component PCA on the activation residuals** the PLS
components leave behind, and writes the scores to CSV with **every prompt-metadata field
preserved** alongside them.

Layer 18 is the choice the redundancy analysis landed on: it is the argmax of held-out target R²
at position -1 in both full sweeps (0.950 and 0.955 mean across datasets), and the per-dataset best
in three of four datasets.

Output per dataset, one row per prompt:

| Columns | What they hold |
|---|---|
| every flattened prompt-metadata field | `task`, `template_id`, `base_value`, `base_unit`, `template_metadata.*`, `task_metadata.*`, … |
| `dataset`, `sample_index`, `batch_file` | provenance back to the cached batch |
| `time_horizon_months`, `log10_time_horizon_months` | the target, in months and as fitted |
| `pls_1` … `pls_6` | the PLS scores |
| `pls_prediction`, `pls_residual` | the model's prediction of the target and its error |
| `residual_pc_1` - `residual_pc_3` | PCA of the activation left unexplained by the PLS components |

The fitted projection is saved as well, so the identical transform can be applied to new
activations later without refitting.

Both `LAYER` and `POSITION` also accept a list - `LAYER = [17, 18]`, `POSITION = [-2, -1]`. Every
layer-position pair is then concatenated along the feature axis before the PLS fit, laid out
layer-major and position-minor, so a prompt is described by `2,560 x len(LAYER) x len(POSITION)`
floats and every component draws on all of them at once. Output filenames carry the layers and the
positions joined by underscores (`..._layer17_18_pos-2_-1_pls6.csv`).

Nothing in the notebook holds a dataset in memory. Activations are staged to an on-disk
memory-mapped array (2,560 floats per prompt per layer-position pair), the fit sample is assembled
on disk as well, and the residual PCA and the CSVs are both streamed - so peak memory is set by
`FIT_MEMORY_BUDGET_GB` and `CHUNK_MEMORY_BUDGET_GB` rather than by how much data was selected. A
dataset
of any size can be scored without holding it in RAM; batches are downloaded, consumed, and deleted
one group at a time.


## 1. Setup

For a fresh Colab runtime, uncomment the clone line.


In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
# %cd temporal-manifolds
# !mv -n .env.example .env


In [ ]:
import gc
import json
import math
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from dotenv import load_dotenv
from google.cloud import storage
from numpy.lib.format import open_memmap
from plotly.subplots import make_subplots
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import IncrementalPCA
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')


## 2. Authenticate to Google Cloud

In Colab, `google.colab.auth.authenticate_user()` installs the Application Default Credentials the
storage client needs; without it the client falls back to the Compute Engine metadata service and
fails with a `RefreshError`. Locally the credentials come from
`gcloud auth application-default login`.


In [ ]:
import google.auth

try:
    from google.colab import auth as colab_auth
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    colab_auth.authenticate_user()
    print('Authenticated through Colab.')

try:
    credentials, detected_project = google.auth.default(
        scopes=['https://www.googleapis.com/auth/devstorage.read_only']
    )
except google.auth.exceptions.DefaultCredentialsError as error:
    raise RuntimeError(
        'No Google Cloud credentials found. In Colab run '
        '`from google.colab import auth; auth.authenticate_user()`; '
        'locally run `gcloud auth application-default login`.'
    ) from error

PROJECT_ID = os.getenv('GCP_PROJECT_ID') or detected_project or 'temporal-interp-exp'
print(f'Credentials: {type(credentials).__name__}')
print(f'Project: {PROJECT_ID}')


## 3. Configure

Layer 18 lives in the `expanded_` caches (layers 17-35), so `RANGE_TAG` points there. Naming several
layers in `LAYER` only works for layers held in that same cache folder, since one batch file is the
only thing read.

`task_only` is absent from `DATASETS` because its prompts declare no time horizon - the target
would be undefined for every row.

`FIT_DATASETS` chooses which datasets the shared PLS model learns from, independently of which
datasets get scored. Leave it `None` to fit on all of them; name a subset to fit there and score the
rest as unseen data - for example `FIT_DATASETS = ['conversational']` fits on the
conversational phrasing and reports how that axis transfers to `abstract` and the `plain_*` sets. It
applies only when `SHARED_MODEL` is true; per-dataset models always fit on their own dataset.

**On cost:** each cached batch file holds all nineteen layers at both positions (~25 MB) even
though only one layer at one position is used, so scoring a dataset in full means downloading its
whole folder. `conversational` is 2,618 files (~65 GB). Set `BATCH_SAMPLE_SIZE` to score a
spread-out subset instead of everything, and
`MAX_ROWS_PER_BATCH` to thin each file.


In [ ]:
BUCKET_NAME = 'temporal-research-bucket'
RANGE_TAG = 'expanded_'                 # Folder tag holding layers 17-35.
LAYER: int | list[int] = 18             # Layer(s) to score; a list concatenates them.
POSITION: int | list[int] = -1          # Prompt token(s) to score; a list concatenates them.
PLS_COMPONENTS = 6
RESIDUAL_PCA_COMPONENTS = 3             # PCA components fitted on the PLS activation residuals.


@dataclass(frozen=True)
class Dataset:
    name: str
    gcs_suffix: str


DATASETS = [
    Dataset('conversational', 'selected_acts'),
    Dataset('abstract', 'abstract_selected_acts'),
    Dataset('plain_english', 'plain_english_selected_acts'),
    Dataset('plain_long', 'plain_long_selected_acts'),
    # Dataset('conversational', 'selected_acts'),   # ~65 GB; the NOF twin covers the same prompts.
]

DATA_DIR = repo_root / 'data' / 'layer18_pls'
OUTPUT_DIR = repo_root / 'results' / 'layer18_pls'

BATCH_SAMPLE_SIZE: int | None = None    # Batch files to score, evenly spaced; None takes every file.
MAX_ROWS_PER_BATCH: int | None = None   # Rows kept per file; None keeps all 128.
DOWNLOAD_WORKERS = 8
FIT_SAMPLE_SIZE = 50_000                # Rows the PLS model is fitted on; None fits on everything.
FIT_MEMORY_BUDGET_GB = 6.0              # RAM the fit may use; the row count is cut to respect it.
CHUNK_MEMORY_BUDGET_GB = 0.5            # RAM one streamed chunk may use.
SHARED_MODEL = True                     # One model across datasets, so scores are comparable.
FIT_DATASETS: list[str] | None = None   # Dataset names the shared model learns from; None uses all.
INCLUDE_PROMPTS = False                 # Write the full prompt text into the CSV.
FEATURE_CHUNK_ROWS = 4_096              # Upper bound on rows per chunk; the budget above may lower it.
CSV_CHUNK_ROWS = 50_000                 # Rows per append when writing the CSVs.
RANDOM_SEED = 0

LAYERS = [LAYER] if isinstance(LAYER, int) else list(LAYER)
if not LAYERS:
    raise ValueError('LAYER must name at least one layer.')
if len(set(LAYERS)) != len(LAYERS):
    raise ValueError(f'LAYER repeats a layer: {LAYERS}')
LAYER_COMPONENTS = [f'layer_out/{layer}' for layer in LAYERS]
LAYER_TAG = '_'.join(str(layer) for layer in LAYERS)

POSITIONS = [POSITION] if isinstance(POSITION, int) else list(POSITION)
if not POSITIONS:
    raise ValueError('POSITION must name at least one prompt token.')
if len(set(POSITIONS)) != len(POSITIONS):
    raise ValueError(f'POSITION repeats a token: {POSITIONS}')
POSITION_TAG = '_'.join(str(position) for position in POSITIONS)
DATASET_NAMES = [dataset.name for dataset in DATASETS]
unknown_fit_datasets = sorted(set(FIT_DATASETS or []) - set(DATASET_NAMES))
if unknown_fit_datasets:
    raise ValueError(
        f'FIT_DATASETS names datasets that are not being scored: {unknown_fit_datasets}. '
        f'Available: {DATASET_NAMES}'
    )
FIT_DATASET_NAMES = list(FIT_DATASETS) if FIT_DATASETS else DATASET_NAMES

rng = np.random.default_rng(RANDOM_SEED)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
client = storage.Client(project=PROJECT_ID, credentials=credentials)
layer_label = (
    LAYER_COMPONENTS[0] if len(LAYERS) == 1 else f'layer_out {LAYERS} concatenated'
)
position_label = (
    f'position {POSITIONS[0]}' if len(POSITIONS) == 1
    else f'positions {POSITIONS} concatenated'
)
print(f'Scoring {layer_label} at {position_label} for {len(DATASETS)} dataset(s).')
if SHARED_MODEL:
    held_out_datasets = [name for name in DATASET_NAMES if name not in FIT_DATASET_NAMES]
    print(f'Shared model fitted on: {FIT_DATASET_NAMES}')
    if held_out_datasets:
        print(f'Scored but not fitted on: {held_out_datasets} (their R2 is an out-of-distribution estimate)')


## 4. Helpers

Horizon conversion matches `scripts/fit_expanded_pls_residual_pca.py` and the
`download_filtered_*` notebooks, so `log10_time_horizon_months` means the same thing everywhere.


In [ ]:
UNIT_TO_MONTHS = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1.0, 'year': 12.0, 'decade': 120.0, 'century': 1200.0, 'millennium': 12000.0,
}
UNIT_TO_MONTHS.update({f'{unit}s': value for unit, value in list(UNIT_TO_MONTHS.items())})
UNIT_TO_MONTHS['centuries'] = 1200.0
UNIT_TO_MONTHS['millennia'] = 12000.0


def horizon_months(metadata):
    """Return a prompt's horizon in months, or None when it declares none."""
    value = metadata.get('base_value', metadata.get('value'))
    unit = metadata.get('base_unit', metadata.get('unit'))
    if value in (None, 'N/A') or unit in (None, 'N/A'):
        return None
    unit_key = str(unit).lower()
    if unit_key not in UNIT_TO_MONTHS:
        raise ValueError(f'Cannot convert time-horizon unit {unit!r} to months.')
    months = float(value) * UNIT_TO_MONTHS[unit_key]
    return months if months > 0 else None


def flatten_metadata(metadata):
    """Flatten one prompt's nested metadata into dotted scalar fields."""
    flattened = {}
    for key, value in metadata.items():
        if isinstance(value, dict):
            flattened.update({f'{key}.{inner}': item for inner, item in value.items()})
        elif not isinstance(value, (list, tuple, set)):
            flattened[key] = value
    return flattened


def folder_prefix(dataset):
    return f'{RANGE_TAG}{dataset.gcs_suffix}'


def list_batch_blobs(dataset):
    """Every activation batch blob in one dataset folder, ordered by name."""
    prefix = folder_prefix(dataset)
    blobs = sorted(
        (
            blob for blob in client.bucket(BUCKET_NAME).list_blobs(prefix=prefix + '/')
            if Path(blob.name).name.startswith('activations_batch_') and blob.name.endswith('.pt')
        ),
        key=lambda blob: blob.name,
    )
    if not blobs:
        raise FileNotFoundError(f'No activation batches below gs://{BUCKET_NAME}/{prefix}')
    if BATCH_SAMPLE_SIZE is None or len(blobs) <= BATCH_SAMPLE_SIZE:
        return blobs
    picks = np.linspace(0, len(blobs) - 1, BATCH_SAMPLE_SIZE).round().astype(int)
    sampled = [blobs[position] for position in dict.fromkeys(picks.tolist())]
    print(f'  sampling {len(sampled)} of {len(blobs)} batch files, evenly spaced.')
    return sampled


def pls_residuals(model, features):
    """The part of each activation the PLS components leave unexplained.

    ``inverse_transform`` maps the scores back into activation space, so the difference is
    exactly what deflation removed - matching `scripts/fit_expanded_pls_residual_pca.py`.
    """
    return features - model.inverse_transform(model.transform(features))


def selected_features(payload):
    """Return this batch's rows for every LAYER_COMPONENTS x POSITIONS pair, concatenated.

    One layer at one position is the plain (rows, hidden) block; otherwise the blocks are laid
    end to end along the feature axis, layer-major and position-minor - so with layers [17, 18]
    and positions [-2, -1] the columns run 17/-2, 17/-1, 18/-2, 18/-1 - giving
    (rows, hidden * len(LAYERS) * len(POSITIONS)).
    """
    activations = payload['activations']
    missing_layers = [name for name in LAYER_COMPONENTS if name not in activations]
    if missing_layers:
        available = sorted(activations)
        raise KeyError(
            f'{missing_layers} not in this batch; it holds {available[0]}..{available[-1]}.'
        )
    cached = list(payload['positions'])
    missing_positions = [position for position in POSITIONS if position not in cached]
    if missing_positions:
        raise ValueError(f'This batch cached positions {cached}; it is missing {missing_positions}.')
    blocks = [
        activations[name][:, cached.index(position), :].to(torch.float32)
        for name in LAYER_COMPONENTS
        for position in POSITIONS
    ]
    return blocks[0] if len(blocks) == 1 else torch.cat(blocks, dim=1)


## 5. Stage one dataset

Batch files are fetched `DOWNLOAD_WORKERS` at a time, the rows for the configured layer(s) at the
configured position(s) are copied - concatenated feature-wise when several are named -
into a memory-mapped `.npy` array, and the `.pt` files are deleted before the next group. Rows whose
prompt declares no horizon are dropped, since the target is undefined for them.


In [ ]:
def stage_dataset(dataset):
    """Download one dataset and return (memmap path, metadata frame, target array)."""
    blobs = list_batch_blobs(dataset)
    download_dir = DATA_DIR / 'batches' / dataset.name
    download_dir.mkdir(parents=True, exist_ok=True)
    features_path = DATA_DIR / f'{dataset.name}_layer{LAYER_TAG}_pos{POSITION_TAG}.npy'

    array = None
    hidden_size = 0
    capacity = 0
    write_cursor = 0
    records = []
    horizons = []
    dropped = 0

    def fetch(blob):
        local_path = download_dir / Path(blob.name).name
        blob.download_to_filename(str(local_path))
        return local_path

    progress = tqdm(total=len(blobs), desc=f'{dataset.name}: staging', leave=False)
    executor = ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS)
    try:
        for group_start in range(0, len(blobs), DOWNLOAD_WORKERS):
            group = blobs[group_start:group_start + DOWNLOAD_WORKERS]
            for local_path in list(executor.map(fetch, group)):
                try:
                    payload = torch.load(local_path, map_location='cpu', weights_only=True, mmap=True)
                    features = selected_features(payload)
                    metadata_rows = payload['prompt_metadata']
                    if len(metadata_rows) != features.shape[0]:
                        raise ValueError(f'{local_path.name}: metadata and activations disagree on rows.')

                    keep, keep_horizons = [], []
                    for offset, metadata in enumerate(metadata_rows):
                        months = horizon_months(metadata)
                        if months is None:
                            dropped += 1
                            continue
                        keep.append(offset)
                        keep_horizons.append(months)
                    if MAX_ROWS_PER_BATCH is not None and len(keep) > MAX_ROWS_PER_BATCH:
                        chosen = np.sort(rng.choice(len(keep), size=MAX_ROWS_PER_BATCH, replace=False))
                        keep = [keep[position] for position in chosen]
                        keep_horizons = [keep_horizons[position] for position in chosen]
                    if not keep:
                        continue

                    if array is None:
                        hidden_size = int(features.shape[1])
                        capacity = len(blobs) * (MAX_ROWS_PER_BATCH or int(features.shape[0]))
                        array = open_memmap(features_path, mode='w+', dtype=np.float32,
                                            shape=(capacity, hidden_size))
                    take = min(len(keep), capacity - write_cursor)
                    if take <= 0:
                        continue
                    keep, keep_horizons = keep[:take], keep_horizons[:take]
                    array[write_cursor:write_cursor + take] = (
                        features[torch.as_tensor(keep, dtype=torch.long)].numpy()
                    )

                    for offset, months in zip(keep, keep_horizons):
                        record = flatten_metadata(metadata_rows[offset])
                        record['dataset'] = dataset.name
                        record['sample_index'] = int(payload['sample_indices'][offset])
                        record['batch_file'] = local_path.name
                        record['time_horizon_months'] = months
                        record['log10_time_horizon_months'] = float(np.log10(months))
                        if INCLUDE_PROMPTS:
                            record['prompt'] = payload['prompts'][offset]
                        records.append(record)
                        horizons.append(months)
                    write_cursor += take
                    del payload, features
                finally:
                    local_path.unlink(missing_ok=True)
                    progress.update(1)
    finally:
        executor.shutdown(wait=True)
        progress.close()
        shutil.rmtree(download_dir, ignore_errors=True)
        if array is not None:
            array.flush()
        del array
        gc.collect()

    if write_cursor == 0:
        raise ValueError(f'{dataset.name}: no rows with a time horizon were staged.')
    if dropped:
        print(f'  dropped {dropped:,} row(s) without a time horizon.')

    metadata_df = pd.DataFrame(records).reset_index(drop=True)
    target = np.log10(np.array(horizons, dtype=np.float64))
    print(f'  staged {write_cursor:,} rows x {hidden_size:,} features -> {features_path.name}')
    return features_path, write_cursor, metadata_df, target


def open_features(features_path, row_count):
    """Open a staged feature array read-only, sliced to the rows actually written."""
    return np.load(features_path, mmap_mode='r')[:row_count]


## 6. Stage every dataset


In [ ]:
staged = {}
for dataset in DATASETS:
    print(f'=== {dataset.name} (gs://{BUCKET_NAME}/{folder_prefix(dataset)}) ===')
    features_path, row_count, metadata_df, target = stage_dataset(dataset)
    staged[dataset.name] = {
        'features_path': features_path, 'row_count': row_count,
        'metadata': metadata_df, 'target': target,
    }

total_rows = sum(entry['row_count'] for entry in staged.values())
print(f'Staged {total_rows:,} rows across {len(staged)} dataset(s).')


## 7. Fit the PLS model

With `SHARED_MODEL = True` one model is fitted on a pooled sample from the datasets named in
`FIT_DATASETS`, so the six scores mean the same thing in every output file and rows can be compared
across datasets. Set `SHARED_MODEL = False` for a per-dataset model instead, in which case
`FIT_DATASETS` does not apply.

The fit sample is assembled into a memory-mapped file rather than concatenated in RAM, because
scikit-learn casts it to float64 internally and keeps a deflation copy beside it - roughly three
float64 copies of the sample, which is what `FIT_MEMORY_BUDGET_GB` bounds. When `FIT_SAMPLE_SIZE`
asks for more rows than that budget allows, the row count is cut and the reduction is printed;
raise the budget on a machine with room to spare.

A `RESIDUAL_PCA_COMPONENTS`-component PCA is then fitted on the same rows, on what deflation left
over - `X - inverse_transform(transform(X))`, the part of each activation the horizon axis does not
explain. Those directions are orthogonal to the PLS scores by construction, so they describe the
structure that survives once the horizon signal is removed. The printout reports how much of the
centered activation energy the PLS components left behind and how much of that the three PCs pick
up; both land in the diagnostics table and JSON.

`IncrementalPCA` does that work batch by batch, so the residual matrix - another full copy of the
sample - never exists. Its components match a one-shot `PCA` to within batching error, and it is
what lets the fit run at a fixed memory ceiling regardless of how wide the feature vector got.

The model is fitted on at most `FIT_SAMPLE_SIZE` rows drawn at random and split evenly across the
fitting datasets. R² is reported on the rows held out of the fit; for a dataset excluded from
`FIT_DATASETS` every row is held out, so its R² is a transfer estimate onto phrasing the model never
saw.


In [ ]:
def sample_rows(row_count, budget):
    """Indices of a random subset, or every row when the budget covers it."""
    if budget is None or budget >= row_count:
        return np.arange(row_count)
    return np.sort(rng.choice(row_count, size=budget, replace=False))


def rows_within_budget(feature_width, budget_gb, copies):
    """How many rows fit in `budget_gb` once `copies` float64 copies of them are live."""
    bytes_per_row = feature_width * 8 * copies
    return max(PLS_COMPONENTS + 1, int(budget_gb * 1024 ** 3 // bytes_per_row))


# scikit-learn casts the fit matrix to float64 and keeps a deflation copy beside it, so the fit
# costs about three float64 copies of the sample no matter how the sample is stored.
FIT_COPY_FACTOR = 3
FEATURE_WIDTH = int(np.load(next(iter(staged.values()))['features_path'], mmap_mode='r').shape[1])
FIT_ROW_LIMIT = rows_within_budget(FEATURE_WIDTH, FIT_MEMORY_BUDGET_GB, FIT_COPY_FACTOR)
FIT_ROWS = FIT_ROW_LIMIT if FIT_SAMPLE_SIZE is None else min(FIT_SAMPLE_SIZE, FIT_ROW_LIMIT)
STREAM_CHUNK_ROWS = max(1, min(FEATURE_CHUNK_ROWS,
                               rows_within_budget(FEATURE_WIDTH, CHUNK_MEMORY_BUDGET_GB, 3)))
FIT_MATRIX_PATH = DATA_DIR / f'fit_sample_layer{LAYER_TAG}_pos{POSITION_TAG}.npy'

print(f'Feature width {FEATURE_WIDTH:,}; fitting on at most {FIT_ROWS:,} rows '
      f'(~{FIT_ROWS * FEATURE_WIDTH * 8 * FIT_COPY_FACTOR / 1024 ** 3:.1f} GB of working memory), '
      f'streaming {STREAM_CHUNK_ROWS:,} rows at a time.')
if FIT_SAMPLE_SIZE is not None and FIT_ROWS < FIT_SAMPLE_SIZE:
    print(f'  FIT_SAMPLE_SIZE={FIT_SAMPLE_SIZE:,} was cut to {FIT_ROWS:,} to stay inside '
          f'FIT_MEMORY_BUDGET_GB={FIT_MEMORY_BUDGET_GB:g}. Raise the budget if the machine has room.')


def build_fit_matrix(selection):
    """Copy the chosen rows of each dataset into one on-disk float32 matrix.

    Staging the fit sample on disk rather than concatenating it in RAM means the only large
    in-memory array is the float64 working copy scikit-learn makes inside `fit`.
    """
    total = sum(len(indices) for _, indices in selection)
    matrix = open_memmap(FIT_MATRIX_PATH, mode='w+', dtype=np.float32,
                         shape=(total, FEATURE_WIDTH))
    target = np.empty(total, dtype=np.float64)
    cursor = 0
    for entry, indices in selection:
        features = open_features(entry['features_path'], entry['row_count'])
        for start in range(0, len(indices), STREAM_CHUNK_ROWS):
            block = indices[start:start + STREAM_CHUNK_ROWS]
            matrix[cursor:cursor + len(block)] = features[block]
            cursor += len(block)
        target[cursor - len(indices):cursor] = entry['target'][indices]
        del features
    matrix.flush()
    gc.collect()
    return matrix, target


def fit_pls(matrix, target):
    """Fit PLS on an on-disk fit matrix, then an incremental PCA on the residuals it leaves.

    Everything after `fit` streams the matrix a chunk at a time - the R2, the activation mean,
    the energy accounting, and the residual PCA - so peak memory stays near one chunk instead
    of several whole-sample copies. `IncrementalPCA` replaces `PCA` for the same reason: it
    consumes the residuals batch by batch and never holds the residual matrix.

    Returns the PLS model, the residual PCA, the PLS R2 on the fit sample, and the fraction of
    the centered activation energy the PLS components did not explain.
    """
    rows = matrix.shape[0]
    model = PLSRegression(n_components=PLS_COMPONENTS, scale=False)
    model.fit(matrix, target)
    gc.collect()

    predictions = np.empty(rows, dtype=np.float64)
    feature_sum = np.zeros(FEATURE_WIDTH, dtype=np.float64)
    for start in range(0, rows, STREAM_CHUNK_ROWS):
        block = np.asarray(matrix[start:start + STREAM_CHUNK_ROWS], dtype=np.float32)
        predictions[start:start + len(block)] = model.predict(block).reshape(-1)
        feature_sum += block.sum(axis=0, dtype=np.float64)
        del block
    fit_r2 = float(
        1.0 - np.square(target - predictions).sum() / np.square(target - target.mean()).sum()
    )
    feature_mean = (feature_sum / rows).astype(np.float32)
    del predictions, feature_sum

    residual_model = IncrementalPCA(n_components=RESIDUAL_PCA_COMPONENTS)
    centered_energy = 0.0
    residual_energy = 0.0
    for start in tqdm(range(0, rows, STREAM_CHUNK_ROWS), desc='residual PCA', leave=False):
        block = np.asarray(matrix[start:start + STREAM_CHUNK_ROWS], dtype=np.float32)
        residuals = pls_residuals(model, block)
        block -= feature_mean
        centered_energy += float(np.square(block).sum())
        residual_energy += float(np.square(residuals).sum())
        if len(residuals) >= RESIDUAL_PCA_COMPONENTS:
            residual_model.partial_fit(residuals)
        del block, residuals
        gc.collect()
    return model, residual_model, fit_r2, residual_energy / centered_energy


def describe_fit(label, fit_r2, residual_model, residual_energy):
    """Print the PLS R2 alongside what the residual PCA found."""
    explained = ', '.join(f'{value:.3f}' for value in residual_model.explained_variance_ratio_)
    print(f'{label} R2 on the fit sample: {fit_r2:.4f}')
    print(f'  residual energy left by PLS: {residual_energy:.4f} of the centered total; '
          f'residual PCA explains [{explained}] of it')


models = {}
residual_models = {}
fit_stats = {}
fit_indices = {}
if SHARED_MODEL:
    fitting = [name for name in FIT_DATASET_NAMES if name in staged]
    if not fitting:
        raise ValueError(f'None of FIT_DATASETS {FIT_DATASET_NAMES} were staged successfully.')
    per_dataset_budget = max(PLS_COMPONENTS + 1, FIT_ROWS // len(fitting))
    selection = []
    for name, entry in staged.items():
        if name not in fitting:
            fit_indices[name] = np.array([], dtype=int)
            print(f'{name}: excluded from the fit; all {entry["row_count"]:,} rows are held out')
            continue
        indices = sample_rows(entry['row_count'], per_dataset_budget)
        fit_indices[name] = indices
        selection.append((entry, indices))
        print(f'{name}: {len(indices):,} rows contributed to the shared fit')
    fit_matrix, fit_target = build_fit_matrix(selection)
    shared_model, shared_residual_model, shared_fit_r2, shared_residual_energy = fit_pls(
        fit_matrix, fit_target
    )
    del fit_matrix, fit_target, selection
    gc.collect()
    models = {name: shared_model for name in staged}
    residual_models = {name: shared_residual_model for name in staged}
    fit_stats = {name: {'fit_r2': shared_fit_r2, 'residual_energy_fraction': shared_residual_energy}
                 for name in staged}
    describe_fit('Shared model fitted;', shared_fit_r2, shared_residual_model,
                 shared_residual_energy)
else:
    for name, entry in staged.items():
        indices = sample_rows(entry['row_count'], FIT_ROWS)
        fit_indices[name] = indices
        fit_matrix, fit_target = build_fit_matrix([(entry, indices)])
        models[name], residual_models[name], fit_r2, residual_energy = fit_pls(
            fit_matrix, fit_target
        )
        fit_stats[name] = {'fit_r2': fit_r2, 'residual_energy_fraction': residual_energy}
        describe_fit(f'{name}: fitted on {len(indices):,} rows;', fit_r2,
                     residual_models[name], residual_energy)
        del fit_matrix, fit_target
        gc.collect()

FIT_MATRIX_PATH.unlink(missing_ok=True)


## 8. Score every row and write the CSVs

Scores are computed in chunks straight off the memmap, so the full feature matrix is never
materialized - each chunk is projected onto the PLS components, reconstructed, and the leftover is
projected onto the residual PCs in the same pass. Only the outputs stay in memory, a dozen or so
floats per row.

The CSVs are written the same way: `CSV_CHUNK_ROWS` rows of metadata are joined to their scores and
appended to both the per-dataset file and the combined file, so neither a full output frame nor the
concatenation of all of them is ever built. The combined file therefore needs its column order
decided up front, since the datasets do not all carry the same metadata fields - a dataset missing
a field gets an empty cell for it, exactly as the old `pd.concat(..., sort=False)` produced.


In [ ]:
PLOT_SAMPLE = 4_000


def transform_in_chunks(model, residual_model, features_path, row_count, chunk=STREAM_CHUNK_ROWS):
    """Return (PLS scores, predictions, residual PCA scores) for every staged row.

    Only `chunk` rows of activation are in memory at a time. The outputs are a handful of floats
    per row, so they are small enough to keep for the diagnostics below.
    """
    features = open_features(features_path, row_count)
    scores = np.empty((row_count, PLS_COMPONENTS), dtype=np.float32)
    predictions = np.empty(row_count, dtype=np.float32)
    residual_scores = np.empty((row_count, RESIDUAL_PCA_COMPONENTS), dtype=np.float32)
    for start in tqdm(range(0, row_count, chunk), desc='scoring', leave=False):
        block = np.asarray(features[start:start + chunk], dtype=np.float32)
        block_scores = model.transform(block)
        scores[start:start + len(block)] = block_scores.astype(np.float32)
        predictions[start:start + len(block)] = model.predict(block).reshape(-1).astype(np.float32)
        block_residuals = block - model.inverse_transform(block_scores)
        residual_scores[start:start + len(block)] = (
            residual_model.transform(block_residuals).astype(np.float32)
        )
        del block, block_scores, block_residuals
    return scores, predictions, residual_scores


score_columns = [f'pls_{index + 1}' for index in range(PLS_COMPONENTS)]
residual_columns = [f'residual_pc_{index + 1}' for index in range(RESIDUAL_PCA_COMPONENTS)]
derived_columns = score_columns + ['pls_prediction', 'pls_residual'] + residual_columns

# The combined file needs one column order up front, since the datasets do not all carry the same
# metadata fields and the rows are appended a chunk at a time rather than concatenated at the end.
combined_columns = list(dict.fromkeys(
    [column for entry in staged.values() for column in entry['metadata'].columns] + derived_columns
))
combined_path = OUTPUT_DIR / f'all_datasets_layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}.csv'
combined_path.unlink(missing_ok=True)
combined_rows = 0

written = []
diagnostics = []
plot_samples = {}

for name, entry in staged.items():
    model = models[name]
    row_count = entry['row_count']
    target = entry['target']
    scores, predictions, residual_scores = transform_in_chunks(
        model, residual_models[name], entry['features_path'], row_count
    )

    csv_path = OUTPUT_DIR / f'{name}_layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}.csv'
    csv_path.unlink(missing_ok=True)
    for start in range(0, row_count, CSV_CHUNK_ROWS):
        stop = min(start + CSV_CHUNK_ROWS, row_count)
        chunk_frame = entry['metadata'].iloc[start:stop].copy()
        chunk_frame[score_columns] = scores[start:stop]
        chunk_frame['pls_prediction'] = predictions[start:stop]
        chunk_frame['pls_residual'] = (
            chunk_frame['log10_time_horizon_months'] - chunk_frame['pls_prediction']
        )
        chunk_frame[residual_columns] = residual_scores[start:stop]
        chunk_frame.to_csv(csv_path, mode='a', header=start == 0, index=False)
        chunk_frame.reindex(columns=combined_columns).to_csv(
            combined_path, mode='a', header=combined_rows == 0, index=False
        )
        combined_rows += len(chunk_frame)
        del chunk_frame
    written.append(csv_path)

    held_out = np.setdiff1d(np.arange(row_count), fit_indices[name], assume_unique=False)

    def r2(rows):
        if len(rows) < 2:
            return float('nan')
        actual = target[rows]
        residual = actual - predictions[rows]
        return float(1.0 - np.square(residual).sum() / np.square(actual - actual.mean()).sum())

    record = {
        'dataset': name,
        'rows': int(row_count),
        'in_fit_set': bool(len(fit_indices[name])),
        'fit_rows': int(len(fit_indices[name])),
        'held_out_rows': int(len(held_out)),
        'r2_all_rows': r2(np.arange(row_count)),
        'r2_held_out': r2(held_out),
        'pls_1_target_r': float(np.corrcoef(scores[:, 0], target)[0, 1]),
        'residual_energy_fraction': fit_stats[name]['residual_energy_fraction'],
        'residual_pca_explained_variance_ratio': [
            float(value) for value in residual_models[name].explained_variance_ratio_
        ],
        'csv': str(csv_path),
    }
    diagnostics.append(record)
    print(f'{name}: {record["rows"]:,} rows -> {csv_path.name} '
          f'(R2 all {record["r2_all_rows"]:.4f}, held-out {record["r2_held_out"]:.4f})')

    # Keep only what section 10 plots, rather than the whole frame.
    sample = np.sort(rng.choice(row_count, size=min(PLOT_SAMPLE, row_count), replace=False))
    plot_samples[name] = pd.DataFrame({
        'log10_time_horizon_months': target[sample],
        'pls_1': scores[sample, 0],
    })

    del scores, predictions, residual_scores, held_out
    gc.collect()

print(f'Combined: {combined_rows:,} rows -> {combined_path}')
pd.DataFrame(diagnostics).round(4)


## 9. Save the fitted projection

The PLS transform is fully described by the training mean, the rotation matrix, and the regression
coefficients; the residual PCA adds its own mean and components. Saving them means new activations
can be scored later without refitting, and without depending on a pickled scikit-learn version:

```python
loaded = np.load('.../layer18_pls_model.npz')
centered = new_activations - loaded['x_mean']
scores = centered @ loaded['x_rotations']
residuals = centered - scores @ loaded['x_loadings'].T
residual_scores = (residuals - loaded['residual_pc_mean']) @ loaded['residual_pc_components'].T
```


In [ ]:
model_path = OUTPUT_DIR / f'layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}_model.npz'
reference_model = models[next(iter(models))] if SHARED_MODEL else None
reference_residual_model = residual_models[next(iter(residual_models))] if SHARED_MODEL else None

if SHARED_MODEL:
    np.savez(
        model_path,
        x_mean=reference_model._x_mean,
        y_mean=reference_model._y_mean,
        x_rotations=reference_model.x_rotations_,
        x_loadings=reference_model.x_loadings_,
        coef=reference_model.coef_,
        intercept=reference_model.intercept_,
        residual_pc_mean=reference_residual_model.mean_,
        residual_pc_components=reference_residual_model.components_,
        residual_pc_explained_variance_ratio=reference_residual_model.explained_variance_ratio_,
        residual_components=np.array([RESIDUAL_PCA_COMPONENTS]),
        layer=np.array(LAYERS),
        position=np.array(POSITIONS),
        components=np.array([PLS_COMPONENTS]),
    )
    print(f'Wrote {model_path}')
else:
    for name, model in models.items():
        path = OUTPUT_DIR / f'{name}_layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}_model.npz'
        residual_model = residual_models[name]
        np.savez(path, x_mean=model._x_mean, y_mean=model._y_mean,
                 x_rotations=model.x_rotations_, x_loadings=model.x_loadings_,
                 coef=model.coef_, intercept=model.intercept_,
                 residual_pc_mean=residual_model.mean_,
                 residual_pc_components=residual_model.components_,
                 residual_pc_explained_variance_ratio=residual_model.explained_variance_ratio_,
                 residual_components=np.array([RESIDUAL_PCA_COMPONENTS]),
                 layer=np.array(LAYERS), position=np.array(POSITIONS),
                 components=np.array([PLS_COMPONENTS]))
        print(f'Wrote {path}')

metadata_path = OUTPUT_DIR / f'layer{LAYER_TAG}_pos{POSITION_TAG}_pls{PLS_COMPONENTS}_diagnostics.json'
metadata_path.write_text(json.dumps({
    'layer_components': LAYER_COMPONENTS,
    'positions': POSITIONS,
    'pls_components': PLS_COMPONENTS,
    'residual_pca_components': RESIDUAL_PCA_COMPONENTS,
    'shared_model': SHARED_MODEL,
    'fit_datasets': FIT_DATASET_NAMES if SHARED_MODEL else 'per-dataset',
    'bucket': BUCKET_NAME,
    'folders': [folder_prefix(dataset) for dataset in DATASETS],
    'batch_sample_size': BATCH_SAMPLE_SIZE,
    'max_rows_per_batch': MAX_ROWS_PER_BATCH,
    'fit_sample_size': FIT_SAMPLE_SIZE,
    'random_seed': RANDOM_SEED,
    'datasets': diagnostics,
}, indent=2))
print(f'Wrote {metadata_path}')


## 10. Sanity check

The first PLS component should track the target closely. Each dataset gets its own panel, sharing
one axis pair, with a sample of points drawn to keep the figure light.


In [ ]:
INK, MUTED, GRID, MARK = '#0b0b0b', '#52514e', '#e6e5e1', '#2a78d6'

names = list(staged)
columns = min(len(names), 3)
rows = int(np.ceil(len(names) / columns))
fig = make_subplots(rows=rows, cols=columns, subplot_titles=names,
                    horizontal_spacing=0.08, vertical_spacing=0.16)
for index, name in enumerate(names):
    sample = plot_samples[name]
    fig.add_trace(
        go.Scattergl(
            x=sample['log10_time_horizon_months'], y=sample['pls_1'],
            mode='markers', marker={'size': 4, 'color': MARK, 'opacity': 0.45},
            name=name, showlegend=False,
            hovertemplate='log10 months %{x:.2f}<br>PLS-1 %{y:.2f}<extra></extra>',
        ),
        row=index // columns + 1, col=index % columns + 1,
    )
fig.update_layout(
    title={'text': f'PLS component 1 against the target<br>'
                   f'<sup>{layer_label} at {position_label}; '
                   f'{PLOT_SAMPLE:,} points sampled per dataset</sup>',
           'font': {'size': 17, 'color': INK}, 'x': 0, 'xanchor': 'left'},
    template='simple_white', height=320 * rows + 120,
    margin={'l': 70, 'r': 30, 't': 100, 'b': 60},
    font={'color': MUTED, 'size': 12},
)
fig.update_xaxes(title_text='log10 time horizon (months)', showgrid=False, linecolor=GRID)
fig.update_yaxes(title_text='PLS-1', gridcolor=GRID, linecolor=GRID)
for annotation in fig.layout.annotations:
    annotation.font.size = 12
    annotation.font.color = INK
fig.show()

print('Correlation of PLS-1 with the target (every row, not just the plotted sample):')
for record in diagnostics:
    print(f'  {record["dataset"]:34s} r = {record["pls_1_target_r"]:+.4f}')


## 11. Clean up the staged features

The memmaps are kept so the notebook can be re-run - a different component count, a per-dataset
model - without downloading again. They are the price of the low memory ceiling: budget disk for
roughly `rows x 2,560 x 4 bytes` per layer-position pair, per dataset. Run this cell when finished.


In [ ]:
# shutil.rmtree(DATA_DIR, ignore_errors=True)
# print(f'Removed {DATA_DIR}')
